In [84]:
import os
import sys
import pandas as pd
import argparse
from datetime import datetime, timedelta
import pathlib

# 新增：將上一層目錄加入 sys.path
parent_path = pathlib.Path().absolute().parent
sys.path.insert(0, str(parent_path))
# Setup Django environment
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'website_configs.settings')
import django
django.setup()
# 重要：設定環境變數以允許在 Jupyter 的異步環境中執行同步操作
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"



# NewsData Model Structure

Let's first examine the structure of our NewsData model to understand its fields.

In [58]:
# Now we can import Django models
from app_user_keyword_db.models import NewsData

In [59]:
# Inspect the NewsData model fields
print('NewsData Model Fields:')
for field in NewsData._meta.get_fields():
    print(f'- {field.name}: {field.get_internal_type()}')

# Display total count of news entries
print(f'\nTotal news entries in database: {NewsData.objects.count()}')

NewsData Model Fields:
- item_id: CharField
- date: DateField
- category: CharField
- title: TextField
- content: TextField
- sentiment: FloatField
- top_key_freq: TextField
- tokens: TextField
- tokens_v2: TextField
- entities: TextField
- token_pos: TextField
- link: CharField
- photo_link: CharField

Total news entries in database: 141


# filter_database_fullText

In [60]:

"""
# 使用 Q 物件實現 AND 邏輯
q_objects = Q()
for kw in user_keywords:
    # 注意這裡使用 &= 而不是 |=，因為我們需要 AND 邏輯
    if q_objects == Q():  # 如果是第一個條件
        q_objects = Q(content__contains=kw)
    else:
        q_objects &= Q(content__contains=kw)
"""

"""
# 步驟 1：生成 q_list
q_list = [
    Q(content__contains="台積電"), 
    Q(content__contains="晶圓"), 
    Q(content__contains="半導體")
]

# 步驟 2：使用 reduce 組合
# 過程：
# 第一輪：Q(content__contains="台積電") & Q(content__contains="晶圓")
# 第二輪：結果 & Q(content__contains="半導體")
# 最終結果：Q(content__contains="台積電") & Q(content__contains="晶圓") & Q(content__contains="半導體")

final_q = reduce(and_, q_list)
queryset = queryset.filter(final_q)


and_ 和 and 的差別及使用原因
在 Python 中，and_ 和 and 有著根本性的差異，這就是為什麼在 reduce 函數中使用 and_ 而不是 and：

原因解釋
and 是 Python 關鍵字：

and 是 Python 的內置關鍵字運算符
它不能作為函數被傳遞給其他函數
它直接在表達式中使用：condition1 and condition2
and_ 是函數：

and_ 來自 operator 模組，是一個函數
它接受兩個參數並執行邏輯與操作：and_(a, b) 等價於 a & b
它可以作為參數傳遞給其他函數，例如 reduce

"""

'\n# 步驟 1：生成 q_list\nq_list = [\n    Q(content__contains="台積電"), \n    Q(content__contains="晶圓"), \n    Q(content__contains="半導體")\n]\n\n# 步驟 2：使用 reduce 組合\n# 過程：\n# 第一輪：Q(content__contains="台積電") & Q(content__contains="晶圓")\n# 第二輪：結果 & Q(content__contains="半導體")\n# 最終結果：Q(content__contains="台積電") & Q(content__contains="晶圓") & Q(content__contains="半導體")\n\nfinal_q = reduce(and_, q_list)\nqueryset = queryset.filter(final_q)\n\n\nand_ 和 and 的差別及使用原因\n在 Python 中，and_ 和 and 有著根本性的差異，這就是為什麼在 reduce 函數中使用 and_ 而不是 and：\n\n原因解釋\nand 是 Python 關鍵字：\n\nand 是 Python 的內置關鍵字運算符\n它不能作為函數被傳遞給其他函數\n它直接在表達式中使用：condition1 and condition2\nand_ 是函數：\n\nand_ 來自 operator 模組，是一個函數\n它接受兩個參數並執行邏輯與操作：and_(a, b) 等價於 a & b\n它可以作為參數傳遞給其他函數，例如 reduce\n\n'

In [68]:
from django.db.models import Q, Max


In [69]:
# Searching keywords from "content" column
# This function now uses database queries instead of pandas
def filter_database_fullText(user_keywords, cond, cate, weeks):
    # Get the latest date in the database
    latest_date = NewsData.objects.aggregate(max_date=Max('date'))['max_date']
    # Calculate start date
    start_date = latest_date - timedelta(weeks=weeks)
    
    # Base query - filter by date range
    queryset = NewsData.objects.filter(date__gte=start_date, date__lte=latest_date)
    
    # Filter by category if not "全部"
    if cate != "全部":
        queryset = queryset.filter(category=cate)
    
    # Filter by keywords based on condition (AND or OR)
    if cond == 'and':
        # For AND condition, we need all keywords to be present
        for kw in user_keywords:
            queryset = queryset.filter(content__contains=kw)
    elif cond == 'or':
        # For OR condition, any keyword can be present
        q_objects = Q()
        for kw in user_keywords:
            q_objects |= Q(content__contains=kw)
        # q_objects會長這樣： Q(content__contains=kw1) | Q(content__contains=kw2) | Q(content__contains=kw3)
        # 這樣就可以用在filter裡面了
        queryset = queryset.filter(q_objects)
    
    return queryset

In [72]:

user_keywords = ['川普', '伊朗']  # Example keywords
cond = 'or'  # Example condition (and/or), 
cate = '全球'  # Example category (or "全部" for all categories)
weeks = 15  # Example weeks
queryset = filter_database_fullText(user_keywords, cond, cate, weeks)

In [73]:
for news in queryset:
    print(news.title)

伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺
蘋果MacBook Neo賣太好...爆晶片供應問題 要怎麼解決？
台灣人均GDP估五年內將贏南韓逾1萬美元 韓媒點出關鍵原因
京都男童命案 日媒：繼父曾手機搜尋「如何棄屍」
伊朗前領袖身亡逾1個月未下葬！專家挑明3原因「既弱又怕還交代不了」
伊朗前領袖身亡逾1個月未下葬！專家挑明3原因「既弱又怕還交代不了」
外媒曝川普為伊朗戰爭超焦慮！怒吼官員數小時 腦中重播1危機畫面
伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺
伊朗突又封閉荷莫茲海峽 攻擊2油輪 川普警告「無法勒索美國」
媒體傳話？美伊若要求 中國願接手濃縮鈾
京都男童命案 日媒：繼父曾手機搜尋「如何棄屍」
以黎停火 以複製加薩模式駐防 設「黃線」禁返黎南55村莊
日艦穿越台灣海峽 中全程監控、解放軍東海軍演反制
當了首相得避嫌 高市早苗顧慮中韓 不參拜靖國神社
從戰友到決裂 美總統與教宗互批摧毀百年戰略夥伴關係
「紅色時尚女王」王奇琦 用東方美學驚豔世界
伊麗莎白二世官方傳記確定出版 女歷史學家執筆
翻桌率就是生命 日本拉麵店「不准滑手機」掀討論
逛超市都要帶計算機…53歲記者受夠低薪 改當郵差年收翻快3倍


In [74]:
# Limit to k results and get specific fields
news_items = queryset.values('category', 'title', 'link', 'photo_link')[:3]

In [75]:
news_items

<QuerySet [{'category': '全球', 'title': '伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺', 'link': 'https://udn.com', 'photo_link': 'https://via.placeholder.com/150'}, {'category': '全球', 'title': '蘋果MacBook Neo賣太好...爆晶片供應問題 要怎麼解決？', 'link': 'https://udn.com', 'photo_link': 'https://via.placeholder.com/150'}, {'category': '全球', 'title': '台灣人均GDP估五年內將贏南韓逾1萬美元 韓媒點出關鍵原因', 'link': 'https://udn.com', 'photo_link': 'https://via.placeholder.com/150'}]>

In [76]:
list(queryset.values_list('date', flat=True)) # flat=True to get a flat list 否則會是 tuple

[datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19),
 datetime.date(2026, 4, 19)]

In [77]:
latest_date = NewsData.objects.aggregate(max_date=Max('date'))['max_date']

In [78]:
latest_date

datetime.date(2026, 6, 17)

In [79]:
from django.db.models import Avg, Count, Min, Max, Sum
from django.db.models.functions import Extract

# Aggregate是用來計算總和、平均值、最小值、最大值等統計數據的函數
# 這些函數可以用來對查詢集進行聚合計算，並返回一個字典，其中包含計算結果的鍵值對
# Basic aggregation examples
aggregations = NewsData.objects.aggregate(
    count=Count('item_id'),
    oldest=Min('date'),
    newest=Max('date'),
)
print("Basic aggregations:")
for key, value in aggregations.items():
    print(f"- {key}: {value}")


Basic aggregations:
- count: 141
- oldest: 2026-04-19
- newest: 2026-06-17


In [80]:

# Category-based aggregation
category_counts = NewsData.objects.values('category').annotate(
    count=Count('item_id')
).order_by('-count')[:5]  # Top 5 categories
print("\nTop 5 categories by count:")
for item in category_counts:
    print(f"- {item['category']}: {item['count']} articles")


Top 5 categories by count:
- 運動: 20 articles
- 要聞: 20 articles
- 社會: 20 articles
- 產經: 20 articles
- 地方: 20 articles


In [81]:
# annotate是用來在查詢集中添加計算字段的函數
# 這些計算字段可以用來進行進一步的過濾、排序或分組操作
# values是用來選擇查詢集中的特定字段的函數 字段就是資料庫中的欄位
# Date-based aggregation (articles per day)
date_counts = NewsData.objects.values('date').annotate(
    count=Count('item_id')
).order_by('-date')[:7]  # Last 7 days with data

print("\nArticles per day (most recent 7 days with data):")
for item in date_counts:
    print(f"- {item['date']}: {item['count']} articles")



Articles per day (most recent 7 days with data):
- 2026-06-17: 1 articles
- 2026-04-19: 140 articles


In [82]:
# 將 queryset 轉換為 DataFrame，之後共用其他的資料分析函數
df_query = pd.DataFrame(
    list(
        queryset.values(
            "date",
            "category",
            "title",
            "content",
            "link",
            "photo_link",
            "top_key_freq",
            "sentiment",
        )
    )
)
df_query.head(2)

,date,category,title,content,link,photo_link,top_key_freq,sentiment
0,2026-04-19,全球,伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺,在美伊之間的脆弱停火協議到期的數日前， 伊朗 重新管制對全球能源供應至關重要的荷莫茲海峽（S...,https://udn.com,https://via.placeholder.com/150,"[('伊朗', 7), ('海峽', 5), ('全球', 4), ('荷莫茲', 4), ...",0.96
1,2026-04-19,全球,蘋果MacBook Neo賣太好...爆晶片供應問題 要怎麼解決？,蘋果公司最近推出的MacBook Neo銷售熱度遠超乎蘋果的預期，傳出品質合格的A18 Pr...,https://udn.com,https://via.placeholder.com/150,"[('蘋果', 11), ('晶片', 7), ('問題', 3), ('伊朗', 3), ...",0.96


# Views.py

In [89]:
from django.shortcuts import render
from django.views.decorators.csrf import csrf_exempt
from django.http import JsonResponse
from django.db.models import Q, Max, F
from app_user_keyword_db.models import NewsData

from datetime import datetime, timedelta
import pandas as pd


# 匯入其他 app 的 DataFrame 共用函數
from app_user_keyword_association.views import (
    get_title_link_topk,
    get_related_word_clouddata,
    get_same_para,
)

from app_user_keyword.views import get_keyword_time_based_freq

from app_user_keyword_sentiment.views import get_article_sentiment


# For the key association analysis
def home(request):
    return render(request, "app_user_keyword_db/home.html")


@csrf_exempt
def api_get_userkey_data(request):
    userkey = request.POST.get("userkey")
    cate = request.POST["cate"]  # This is an alternative way to get POST data.
    cond = request.POST.get("cond")
    weeks = int(request.POST.get("weeks"))
    key = userkey.split()

    # Get query results from database
    # 使用 ORM 來過濾資料庫的資料
    # 這裡的 queryset 是一個 QuerySet 對象，包含了符合條件的所有新聞資料
    queryset = filter_database_fullText(key, cond, cate, weeks)
    # 使用原生 SQL 來過濾資料庫的資料 也可以
    # queryset = filter_database_fullText_SQL(key, cond, cate, weeks)

    print(f"Query returned {queryset.count()} results")

    # Check if the result is empty
    if not queryset.exists():  # queryset is not empty
        return JsonResponse({"error": "No results found for the given keywords."})

    # 將 queryset 轉換為 DataFrame，之後共用其他的資料分析函數
    df_query = pd.DataFrame(
        list(
            queryset.values(
                "date",
                "category",
                "title",
                "content",
                "link",
                "photo_link",
                "top_key_freq",
                "sentiment",
            )
        )
    )

    # get the data for showing news articles, related words, and word cloud
    newslinks = get_title_link_topk(df_query, k=10)
    related_words, clouddata = get_related_word_clouddata(df_query)
    same_paragraph = get_same_para(df_query, key, cond, k=6)  # multiple keywords
    # num_articles = queryset.count()  # total number of articles (stories, items)
    num_articles = len(df_query)  # total number of articles (stories, items)

    response_article_info = {
        "newslinks": newslinks,
        "same_paragraph": same_paragraph,
        "clouddata": clouddata,
        "num_articles": num_articles,
    }

    # time-based keyword frequency chart data
    key_time_freq = get_keyword_time_based_freq(df_query)  # OK 使用 pandas 很方便
    response_time_occurrence = {
        "key_time_freq": key_time_freq,
    }

    # piechart for sentiment distribution
    sentiCount, sentiPercnt = get_article_sentiment(df_query)
    response_sentiment = {
        "sentiCount": sentiCount,
    }

    # Combine dictionaries correctly
    combined_response = {
        **response_article_info,
        **response_time_occurrence,
        **response_sentiment,
    }
    return JsonResponse(combined_response)


# Searching keywords from "content" column
# This function now uses database queries instead of pandas
def filter_database_fullText(user_keywords, cond, cate, weeks):
    # def filter_database_fullText_ORM(user_keywords, cond, cate, weeks):
    # Get the latest date in the database
    latest_date = NewsData.objects.aggregate(max_date=Max("date"))["max_date"]

    # Calculate start date
    start_date = latest_date - timedelta(weeks=weeks)

    # Base query - filter by date range
    queryset = NewsData.objects.filter(date__gte=start_date, date__lte=latest_date)

    # Filter by category if not "全部"
    if cate != "全部":
        queryset = queryset.filter(category=cate)

    # Filter by keywords based on condition (AND or OR)
    if cond == "and":
        # For AND condition, we need all keywords to be present
        for kw in user_keywords:
            queryset = queryset.filter(content__contains=kw)
    elif cond == "or":
        # For OR condition, any keyword can be present |  ||
        q_objects = Q()
        for kw in user_keywords:
            q_objects |= Q(content__contains=kw)
        queryset = queryset.filter(q_objects)

    return queryset


print("app_user_keyword_db versions loaded!")

FileNotFoundError: [Errno 2] No such file or directory: 'app_user_keyword/dataset/udn_news_preprocessed.csv'

# Using raw SQL and ORM to count keyword (For ref.)

In [90]:

###################################################################

# 這段程式碼使用原生 SQL 來過濾資料庫的資料
def filter_database_fullText_SQL(user_keywords, cond, cate, weeks):
    # def filter_database_fullText(user_keywords, cond, cate, weeks):
    from django.db import connection

    # Get the latest date using raw SQL
    with connection.cursor() as cursor:
        cursor.execute(f"SELECT MAX(date) FROM {NewsData._meta.db_table}")
        latest_date_str = cursor.fetchone()[0]

    latest_date = datetime.strptime(latest_date_str, "%Y-%m-%d").date()
    # Calculate start date
    start_date = latest_date - timedelta(weeks=weeks)

    # Format dates for SQL query (if necessary)
    start_date_str = start_date.strftime("%Y-%m-%d")
    latest_date_str = latest_date.strftime("%Y-%m-%d")

    # Build the base SQL query
    table_name = NewsData._meta.db_table
    params = [start_date_str, latest_date_str]

    sql = f"""
    SELECT item_id FROM {table_name}
    WHERE date >= %s AND date <= %s
    """

    # Add category filter if not "全部"
    if cate != "全部":
        sql += " AND category = %s"
        params.append(cate)

    # Add keyword filters based on condition
    if user_keywords:
        if cond == "and":
            # For AND, add each keyword as a separate condition
            for kw in user_keywords:
                sql += " AND content LIKE %s"
                params.append(f"%{kw}%")
        elif cond == "or":
            # For OR, combine all keywords with OR
            if user_keywords:
                sql += " AND ("
                or_conditions = []
                for kw in user_keywords:
                    or_conditions.append("content LIKE %s")
                    params.append(f"%{kw}%")
                sql += " OR ".join(or_conditions)
                sql += ")"

    # Execute the raw SQL query to get matching IDs
    with connection.cursor() as cursor:
        cursor.execute(sql, params)
        id_list = [row[0] for row in cursor.fetchall()]

    # Convert back to a queryset for compatibility with the rest of the code
    if id_list:
        queryset = NewsData.objects.filter(item_id__in=id_list)
    else:
        queryset = NewsData.objects.none()  # Empty queryset

    return queryset


# 使用ORM 計算關鍵字頻率和出現次數
# 這段程式碼使用 Django ORM 來計算關鍵字在新聞資料庫中的頻率和出現次數
"""
# ORM Optimization:
Reduced Database Queries:
The ORM efficiently counts articles per category in a single query
This eliminates the need to increment counters for each row individually

Partial ORM Optimization:
Article counts per category are done using Django's values() and annotate(Count())
This moves aggregation work to the database

Limitation:
For keyword frequency counting, we still need Python processing
This is because complex regex pattern matching across multiple keywords can't be easily expressed in pure ORM/SQL
The improved code organizes articles by category first to reduce redundant category lookups

Further Improvements (Optional):
For databases that support regex functions (like PostgreSQL), you could potentially push more of the keyword counting to the database level using annotations, but this would be database-specific and complex. The current approach offers a good balance between optimization and readability.
"""
from django.db.models import Count, F, Func, Value, IntegerField, Sum
from django.db.models.functions import Coalesce


def count_keyword_with_orm(queryset, query_keywords):
    # Dictionary for all categories
    news_categories_with_all = news_categories.copy()

    # Count articles per category (occurrence)
    category_counts = queryset.values("category").annotate(count=Count("item_id"))

    # Initialize dictionaries with zeros
    cate_occurrence = {cate: 0 for cate in news_categories_with_all}
    cate_freq = {cate: 0 for cate in news_categories_with_all}

    # Fill occurrence counts from the query results
    total_count = 0
    for item in category_counts:
        category = item["category"]
        count = item["count"]
        cate_occurrence[category] = count
        total_count += count

    # Set the total count for all categories
    cate_occurrence["全部"] = total_count

    # For frequency counts, we need to use annotations for each keyword
    # This part still requires iterating through news items in Python
    # because regex-based counting across multiple keywords isn't easily
    # expressible in pure ORM

    # Group articles by category
    category_groups = {}
    for row in queryset:
        category = row.category
        if category not in category_groups:
            category_groups[category] = []
        category_groups[category].append(row)

    # Count frequencies by category
    total_freq = 0
    for category, articles in category_groups.items():
        category_freq = 0
        for article in articles:
            # Count keyword occurrences in each article
            freq = sum(
                [
                    len(re.findall(keyword, article.content, re.I))
                    for keyword in query_keywords
                ]
            )
            category_freq += freq

        cate_freq[category] = category_freq
        total_freq += category_freq

    # Set total frequency
    cate_freq["全部"] = total_freq

    return cate_freq, cate_occurrence


"""
Key Advantages of the ORM Approach:
Performance: The aggregation happens at the database level, which is much faster when dealing with large datasets
Memory Efficiency: You're not loading all dates into memory
Less Code: No need for the intermediate pandas DataFrame conversion
Additional Notes:
TruncDate truncates a datetime field to its date component, allowing you to group by day
You can use other truncation functions like TruncHour, TruncWeek, or TruncMonth for different time resolutions
This approach works with any SQL database supported by Django (PostgreSQL, MySQL, SQLite)
The ORM approach is generally preferred when working with Django applications, as it leverages the database's optimization capabilities and maintains consistency with your Django models.
"""
from django.db.models import Count
from django.db.models.functions import TruncDate
from django.db.models import Count
from django.db.models.functions import TruncDate
from django.db.models.functions import Cast
from django.db.models import DateField


def get_keyword_time_based_freq_with_orm(queryset):

    # Use date string extraction (works on SQLite)
    # This uses Cast to convert date field to string in YYYY-MM-DD format
    date_counts = (
        queryset.annotate(day=Cast("date", DateField()))  # Extract just the date part
        .values("day")
        .annotate(count=Count("item_id"))
        .order_by("day")
    )

    time_data = []
    for item in date_counts:
        # Format may vary depending on database
        if isinstance(item["day"], str):
            date_str = item["day"]
        else:
            date_str = item["day"].strftime("%Y-%m-%d")  # Convert to string

        row = {"x": date_str, "y": item["count"]}
        time_data.append(row)

    return time_data


"""
Considerations When Using Raw SQL:
Database Portability: Raw SQL syntax varies between database backends (PostgreSQL, MySQL, SQLite)
Security: Be careful with SQL injection when constructing queries (use parameterized queries)
Maintainability: Raw SQL is harder to maintain than ORM code
Integration with Django: You lose some of Django's automatic handling of connections and transactions
When Raw SQL Might Be Better:
For highly complex queries that are difficult to express in ORM
When you need to use database-specific features or optimizations
When performance is critical and ORM overhead is a concern
"""
import sqlite3
from django.db import connections
import sqlite3


def get_keyword_time_based_freq_raw_sql(queryset):
    if not queryset.exists():
        return []

    table_name = NewsData._meta.db_table
    ids = tuple(queryset.values_list("item_id", flat=True))  # Convert to tuple

    # Handle empty queryset case
    if not ids:
        return []

    # Handle single item case
    if len(ids) == 1:
        ids = (ids[0],)  # Add comma to make it a tuple

    # Fix: Use correct SQL parameterization
    sql = f"""
    SELECT 
        date(date) as day,
        COUNT(*) as count
    FROM 
        {table_name}
    WHERE 
        item_id IN {ids}
    GROUP BY 
        date(date)
    ORDER BY 
        day
    """

    try:
        with connections["default"].cursor() as cursor:
            cursor.execute(sql)  # No need to pass ids separately
            rows = cursor.fetchall()

        time_data = []
        for day, count in rows:
            row = {"x": day, "y": count}
            time_data.append(row)
        return time_data

    except Exception as e:
        print(f"SQL Error: {e}")


"""
有幾項任務確實「非常難」或「不適合」完全依賴資料庫端（SQL）來處理。

我們可以把這段程式碼中的操作分成兩類：「適合在資料庫完成的」與「必須交給 Python 處理的」。

1. 適合且已經在資料庫 (DB) 中完成的任務
Django ORM 在這段程式碼中已經很聰明地把以下工作交給了資料庫：

filter_database_fullText()：條件過濾（日期區間、類別、關鍵字）。這會轉換成 SQL 的 WHERE 與 LIKE 語句，完全在資料庫運作。
queryset.count() 與 queryset.exists()：計算總筆數與檢查是否存在資料。轉換成 SELECT COUNT(*)，非常高效。
get_title_link_topk(queryset, k=10)：取得前 K 筆的特定欄位。看你的程式碼，裡面用了 .values(...)[:k]，這會轉換成 SELECT ... LIMIT 10，也是完全在資料庫做的。
2. 在資料庫中「很難做」或「幾乎不可能做」的任務
這段程式碼中有兩個函式強烈依賴 Python 的邏輯，如果硬要用資料庫 (特別是 SQLite) 來做會非常痛苦甚至做不到：

A. get_same_para (切分段落並找出關鍵字)
為什麼很難在 DB 做？
這項操作需要把一整篇文章的 content 用中文句號 「。」 切割成陣列（List），然後逐一段落使用正則表達式（Regex）去比對是否同時包含所有關鍵字。
標準的 SQL（特別是 SQLite）缺乏強大的字串分割（split）為陣列的功能。即使是 PostgreSQL 有 regexp_split_to_array，要在 SQL 裡面寫迴圈檢查每個段落並將符合條件的段落抽出來，語法會極度複雜且難以維護，效能也不見得好。交給 Python 的 re 與字串處理是最合理的。
B. get_related_word_clouddata (處理文字雲資料與詞頻)
為什麼很難在 DB 做？
從你的原始碼可以看出，news.top_key_freq 欄位存的是像字串一樣的字典（需要用 eval() 或 json.loads 解析）。要在純 SQL 裡把這個字串解析出來、跨多筆資料把相同的詞彙頻率作加總（像是 Python 的 Counter），然後找出前 20 名，再透過公式把頻率轉換成前端文字雲的大小（size 20~120）。
這充滿了商業邏輯與複雜的字串解析，資料庫的強項是集合運算，而不是反序列化字串和視覺化數據轉換。
結論
這是一種很好的架構分工，稱為「胖後端 / 瘦資料庫」：

讓資料庫做它最擅長的事：篩選資料、排序、限制筆數（Limit/Offset）。
讓 Python 做它最擅長的事：複雜字串處理（切句號）、正則表達式（Regex）、字典累加運算（Counter）與商業邏輯轉換。
"""


'\n有幾項任務確實「非常難」或「不適合」完全依賴資料庫端（SQL）來處理。\n\n我們可以把這段程式碼中的操作分成兩類：「適合在資料庫完成的」與「必須交給 Python 處理的」。\n\n1. 適合且已經在資料庫 (DB) 中完成的任務\nDjango ORM 在這段程式碼中已經很聰明地把以下工作交給了資料庫：\n\nfilter_database_fullText()：條件過濾（日期區間、類別、關鍵字）。這會轉換成 SQL 的 WHERE 與 LIKE 語句，完全在資料庫運作。\nqueryset.count() 與 queryset.exists()：計算總筆數與檢查是否存在資料。轉換成 SELECT COUNT(*)，非常高效。\nget_title_link_topk(queryset, k=10)：取得前 K 筆的特定欄位。看你的程式碼，裡面用了 .values(...)[:k]，這會轉換成 SELECT ... LIMIT 10，也是完全在資料庫做的。\n2. 在資料庫中「很難做」或「幾乎不可能做」的任務\n這段程式碼中有兩個函式強烈依賴 Python 的邏輯，如果硬要用資料庫 (特別是 SQLite) 來做會非常痛苦甚至做不到：\n\nA. get_same_para (切分段落並找出關鍵字)\n為什麼很難在 DB 做？\n這項操作需要把一整篇文章的 content 用中文句號 「。」 切割成陣列（List），然後逐一段落使用正則表達式（Regex）去比對是否同時包含所有關鍵字。\n標準的 SQL（特別是 SQLite）缺乏強大的字串分割（split）為陣列的功能。即使是 PostgreSQL 有 regexp_split_to_array，要在 SQL 裡面寫迴圈檢查每個段落並將符合條件的段落抽出來，語法會極度複雜且難以維護，效能也不見得好。交給 Python 的 re 與字串處理是最合理的。\nB. get_related_word_clouddata (處理文字雲資料與詞頻)\n為什麼很難在 DB 做？\n從你的原始碼可以看出，news.top_key_freq 欄位存的是像字串一樣的字典（需要用 eval() 或 json.loads 解析）。要在純 SQL 裡把這個字串解析出來、跨多筆資料把相同的詞彙頻率作加總（像是 Python 的 Co